# Data Clean Up Summary

* First pass for clean up for uploading data to the website
* Import
    * both versions of SGO data (older and newer)
    * treated version of the newer:
        * cleaned up columns TO DO LIST
        * combined narrative
        * potential targets
        * is_latest_of_multiple_report
    * columns that do not exist in old but have analogues in new

In [ ]:
'''
to do
* the is_latest_of_multiple_report
* complete orchestration of all the steps
* list the steps and created columns
'''

'\nto do\n* the is_not_duplicate\n* complete orchestration of all the steps\n* list the steps and created columns\n'

## 0. Setup

In [2]:
import sys
sys.path.append('..')

from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eda_utils_sgo
import eda_utils_dedupe
import eda_utils_treatment
import eda_utils_spacy

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 300)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
ARTIFACTS = Path('artifacts_spacy')
for sub in ['linguistic', 'ner', 'matchers', 'displacy/ent', 'displacy/dep',
            'similarity', '_docbin']:
    (ARTIFACTS / sub).mkdir(parents=True, exist_ok=True)
print('artifact root:', ARTIFACTS.resolve())

artifact root: C:\Users\james\claude_code_repos\avird-2026-eda-v001\eda\ADS_to_2026_03_16\artifacts_spacy


## 1. Load and Treat Data

Same dedupe + treatment preamble as `03_eda_basic_topics_2026.ipynb`. We use the single-report `Narrative` column (not the `Narrative - Same Incident ID` concat) by default -- the `--- next report ---` separator pollutes sentence segmentation.

In [5]:
repo_root = Path.cwd().parents[1]
data_dir = repo_root / 'data' / 'nhtsa'
paths = [
    data_dir / 'SGO-2021-01_Incident_Reports_ADS_to_2025_06_16.csv',
    data_dir / 'SGO-2021-01_Incident_Reports_ADS_2025_06_16_to_2026_03_16.csv',
]
ads_df = eda_utils_sgo.load_and_concat_csvs(paths)

Only in SGO-2021-01_Incident_Reports_ADS_to_2025_06_16.csv:
  ADAS/ADS Hardware Version
  ADAS/ADS Hardware Version - Unk
  ADAS/ADS Hardware Version CBI
  ADAS/ADS Software Version
  ADAS/ADS Software Version - Unk
  ADAS/ADS Software Version CBI
  ADAS/ADS System Version
  ADAS/ADS System Version - Unk
  ADAS/ADS System Version CBI
  ADS Equipped?
  CP Any Air Bags Deployed?
  CP Was Vehicle Towed?
  Federal Reg. Exemption - No
  Federal Reg. Exemption - Unk
  Federal Regulatory Exemption?
  Inv. Officer Email - Unknown
  Inv. Officer Name - Unknown
  Inv. Officer Phone - Unknown
  Investigating Officer Email
  Investigating Officer Name
  Investigating Officer Phone
  Law Enforcement Investigating?
  Lighting
  Mileage
  Mileage - Unknown
  Notice Received Date
  Other Federal Reg. Exemption
  Other Reporting Entities?
  Other Reporting Entities? - NA
  Other Reporting Entities? - Unk
  Posted Speed Limit (MPH)
  Posted Speed Limit - Unknown
  Property Damage?
  Rep Ent Or Mfr Inves

In [8]:
early_df = pd.read_csv(paths[0])
print(early_df.shape)

(2295, 137)


In [9]:
later_df = pd.read_csv(paths[1])
print(later_df.shape)

(825, 116)


In [6]:
entity_column = 'master_entity'
treated_df = eda_utils_dedupe.dedupe_same_incident(ads_df.copy(), verbose=True)
treated_df = eda_utils_treatment.apply_all_treatments(treated_df)
narratives = treated_df['Narrative']
print('rows:', len(treated_df), 'non-null narratives:', narratives.notna().sum())

dedupe_same_incident: 3120 -> 2344 rows (776 duplicates collapsed)
rows: 2344 non-null narratives: 2342


## 2. Examine columns in old that may be analogues in new


In [10]:
early_df.columns.to_list()

['Report ID',
 'Report Version',
 'Reporting Entity',
 'Report Type',
 'Report Month',
 'Report Year',
 'Report Submission Date',
 'VIN',
 'VIN - Unknown',
 'Serial Number',
 'Make',
 'Model',
 'Model - Unknown',
 'Model Year',
 'Model Year - Unknown',
 'Same Vehicle ID',
 'Mileage',
 'Mileage - Unknown',
 'Driver / Operator Type',
 'ADAS/ADS System Version',
 'ADAS/ADS System Version - Unk',
 'ADAS/ADS System Version CBI',
 'ADAS/ADS Hardware Version',
 'ADAS/ADS Hardware Version - Unk',
 'ADAS/ADS Hardware Version CBI',
 'ADAS/ADS Software Version',
 'ADAS/ADS Software Version - Unk',
 'ADAS/ADS Software Version CBI',
 'Other Reporting Entities?',
 'Other Reporting Entities? - Unk',
 'Other Reporting Entities? - NA',
 'Federal Regulatory Exemption?',
 'Other Federal Reg. Exemption',
 'Federal Reg. Exemption - Unk',
 'Federal Reg. Exemption - No',
 'State or Local Permit?',
 'State or Local Permit',
 'ADS Equipped?',
 'Automation System Engaged?',
 'Operating Entity',
 'Operating Enti

In [11]:
later_df.columns.to_list()

['Report ID',
 'Report Version',
 'Reporting Entity',
 'Report Type',
 'Report Month',
 'Report Year',
 'Report Submission Date',
 'VIN',
 'VIN Decoded',
 'Serial Number',
 'Make',
 'Make - Unknown',
 'Model',
 'Model - Unknown',
 'Model Year',
 'Model Year - Unknown',
 'Same Vehicle ID',
 'Driver / Operator Type',
 'Automation Feature Version',
 'Automation Feature Version CBI',
 'Automation System Engaged?',
 'Engagement Status',
 'Operating Entity',
 'Operating Entity - Unknown',
 'Source - Complaint/Claim',
 'Source - Telematics',
 'Source - Law Enforcement',
 'Source - Field Report',
 'Source - Testing',
 'Source - Media',
 'Source - Internal Process Review',
 'Source - NHTSA VOQ',
 'Source - Other - See Narrative',
 'Source - Other Entity',
 'Source - State or Other Agency',
 'Incident Date',
 'Incident Date - Unknown',
 'Incident Time (24:00)',
 'Incident Time - Unknown',
 'Same Incident ID',
 'Latitude',
 'Latitude - Unknown',
 'Longitude',
 'Longitude - Unknown',
 'Address',
 

In [ ]:
AIRBAG_COLS = (
    'Any Air Bags Deployed?',        # newer schema (compound values)
    'CP Any Air Bags Deployed?',     # older schema (simple Yes/No)
    'SV Any Air Bags Deployed?',     # older schema (simple Yes/No)
)
TOWED_COLS = (
    'Was Any Vehicle Towed?',        # newer schema (compound)
    'CP Was Vehicle Towed?',         # older schema (simple Yes/No)
    'SV Was Vehicle Towed?',         # older schema (simple Yes/No) -
                                     # included for completeness; drop via
                                     # the `cols` arg if not wanted.
)



In [ ]:
'Engagement Status' # later
'Automation System Engaged?' # early
'''
Can link ADS to verified engaged, Unknowns, ADAS becomes own thing I guess
'''

# roadway early may have analogues in later
# early:
'Roadway Type',
'Roadway Surface',
'Roadway Description',
# later
'Roadway Type',
'Roadway-Degraded Surface',
'Roadway-Missing/Degraded Marking',
'Roadway-No Unusual Conditions',
'Roadway-Other-See Narrative',
'Roadway-Traffic Incident',
'Roadway-Unknown',
'Roadway-Wet Surface Condition',
'Roadway-Work Zone',

# belted
# later
'Were All Passengers Belted?' # only has subject vehicle
# early
'SV Were All Passengers Belted?',

# weather match
#early
'Weather - Clear',
'Weather - Snow',
'Weather - Cloudy',
'Weather - Fog/Smoke',
'Weather - Rain',
'Weather - Severe Wind',
'Weather - Unknown',
'Weather - Other',
'Weather - Other Text',
# later
'Weather - Clear',
'Weather - Snow',
'Weather - Cloudy',
'Weather - Partly Cloudy',
'Weather - Fog/Smoke/Haze',
'Weather - Rain',
'Weather - Severe Wind',
'Weather - Dust Storm',
'Weather - Severe Hurricane',
'Weather - Structure-Indoor',
'Weather - Unk - See Narrative',

In [12]:
later_df['Engagement Status'].value_counts()

Engagement Status
Verified Engaged           797
Verified Not Engaged        26
Alleged Engaged              1
Unknown - see Narrative      1
Name: count, dtype: int64

In [ ]:
early_df['Automation System Engaged?'].value_counts()

Automation System Engaged?
ADS                       2234
Unknown, see Narrative      30
ADAS                        27
Name: count, dtype: int64

In [14]:
later_df['Were All Passengers Belted?'].value_counts()

Were All Passengers Belted?
Subject Vehicle - All Belted                    411
Subject Vehicle - No Passenger In Vehicle       342
Subject Vehicle - Not Belted - see Narrative     68
Unknown                                           4
Name: count, dtype: int64

In [15]:
early_df['SV Were All Passengers Belted?'].value_counts()

SV Were All Passengers Belted?
Yes                         1339
No Passengers in Vehicle     841
No, see Narrative            101
Unknown                       11
Name: count, dtype: int64

In [16]:
early_df['Lighting'].value_counts()

Lighting
Daylight                   1376
Dark - Lighted              801
Dawn / Dusk                  69
Dark - Not Lighted           32
Unknown                       7
Dark - Unknown Lighting       5
Other, see Narrative          2
Name: count, dtype: int64